In [1]:
import numpy as np
import math
import random
import csv
import pandas as pd
import scipy
import scipy.optimize as opt
from bayes_opt import BayesianOptimization
from bayes_opt.logger import JSONLogger
from bayes_opt.event import Events
from bayes_opt.util import load_logs

In [2]:
def dir(a):
    a_0 = np.sum(a,axis=0)
    A = np.zeros([8,8])
    for i in range(np.shape(A)[0]):
        for j in range(np.shape(A)[1]):
            A[i,j] = a[i,j]/a_0[j]
    return A
def cum(a):
    a_0 = np.sum(a,axis=0)
    a_cum = np.array([np.ones([1,8])*a_0[0],
                      np.ones([1,8])*a_0[1],
                      np.ones([1,8])*a_0[2],
                      np.ones([1,8])*a_0[3],
                      np.ones([1,8])*a_0[4],
                      np.ones([1,8])*a_0[5],
                      np.ones([1,8])*a_0[6],
                      np.ones([1,8])*a_0[7]]).squeeze().T
    return a_cum
def H_entropy(A):
    H = np.matmul(A.T,np.log(A+np.e**(-16)))
    H = np.diag(H)
    return H
def efe_al_ai_ex_ev(A,a,s,action_stay_cue,p_al,p_ai,p_ex):
    a_nonzero = np.array([[1,1,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,0,0,1,1,0,0],
                      [0,0,0,0,0,0,1,0],
                      [0,0,0,0,0,0,0,1]])
    s = np.array(s)
    discount = 0.1
    o = np.dot(A,s.reshape(-1,1)).reshape(-1)
    w = 1./(cum(a))-1./(a+np.e**(-16))
    w = np.multiply(w,a_nonzero)
    H = -np.dot(H_entropy(A),s.reshape(-1,1))
    AL = np.dot(o,np.dot(w,s.reshape(-1,1)))
    AI = H + np.dot(o,np.log(o+np.e**(-16)).reshape(-1,1))
    EV = p_al*AL + p_ai*AI
    preference = np.array([1,2,1.5,0.5,0,0,-0.167,-0.167])
    EX = np.dot(o,preference.reshape(-1,1))
    if action_stay_cue == 0:
        EFE = discount*(p_al*AL + p_ai*AI) - p_ex*EX
        return np.array([EFE,discount*p_al*AL,discount*p_ai*AI,p_ex*EX,discount*EV]).squeeze()
    elif action_stay_cue == 1:
        EFE = p_al*AL + p_ai*AI - p_ex*EX
        return np.array([EFE,p_al*AL,p_ai*AI,p_ex*EX,EV]).squeeze()
    else :
        print('ERROR')
        
def ai(A,a,s,action_stay_cue,p_al,p_ai,p_ex):
    a_nonzero = np.array([[1,1,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,0,0,1,1,0,0],
                      [0,0,0,0,0,0,1,0],
                      [0,0,0,0,0,0,0,1]])
    s = np.array(s)

    o = np.dot(A,s.reshape(-1,1)).reshape(-1)
    w = 1./(cum(a))-1./(a+np.e**(-16))
    w = np.multiply(w,a_nonzero)
    H = -np.dot(H_entropy(A),s.reshape(-1,1))

    AI = H + np.dot(o,np.log(o+np.e**(-16)).reshape(-1,1))
    return AI

def al(A,a,s,action_stay_cue,p_al,p_ai,p_ex):
    a_nonzero = np.array([[1,1,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,0,0,1,1,0,0],
                      [0,0,0,0,0,0,1,0],
                      [0,0,0,0,0,0,0,1]])
    s = np.array(s)
    o = np.dot(A,s.reshape(-1,1)).reshape(-1)
    w = 1./(cum(a))-1./(a+np.e**(-16))
    w = np.multiply(w,a_nonzero)
    AL = np.dot(o,np.dot(w,s.reshape(-1,1)))
    return AL
        
        
def value_stay_cue(A,a,action_stay_cue,if_can_ask,p_al,p_ai,p_ex,norm):
    if if_can_ask == 0:
        return '','','','',''
    else:
        Value_stay_cue = []
        s1 = np.array([0,0,0,0,0.5,0.5,0,0])
        s2 = np.array([0.5,0.5,0,0,0,0,0,0])
        Value_stay_safe = efe_al_ai_ex_ev(A,a,s1,0,p_al,p_ai,p_ex)+efe_al_ai_ex_ev(A,a,s2,0,p_al,p_ai,p_ex)
        s1 = np.array([0,0,0,0,0.5,0.5,0,0])
        s2 = np.array([0,0,0.5,0.5,0,0,0,0])
        Value_stay_risk = efe_al_ai_ex_ev(A,a,s1,0,p_al,p_ai,p_ex)+efe_al_ai_ex_ev(A,a,s2,0,p_al,p_ai,p_ex)
        if Value_stay_safe[0]>Value_stay_risk[0]:
            Value_stay_cue.append(Value_stay_risk)
        else:
            Value_stay_cue.append(Value_stay_safe)        
        s1 = [0, 0, 0, 0, 0, 0, 0.5, 0.5]
        s2 = [1, 0, 0, 0, 0, 0, 0, 0]
        s3 = [0, 0, 0, 0, 0, 0, 0.5, 0.5]
        s4 = [0, 0, 1, 0, 0, 0, 0, 0]
        Value_cue_HR_0 = efe_al_ai_ex_ev(A, a, s1, 1, p_al, p_ai, p_ex) + efe_al_ai_ex_ev(A, a, s2, 1, p_al, p_ai, p_ex)
        Value_cue_HR_1 = efe_al_ai_ex_ev(A, a, s3, 1, p_al, p_ai, p_ex) + efe_al_ai_ex_ev(A, a, s4, 1, p_al, p_ai, p_ex)
        if Value_cue_HR_0[0]>Value_cue_HR_1[0]:
            Value_cue_HR = Value_cue_HR_1
        else:
            Value_cue_HR = Value_cue_HR_0
        s1 = [0, 0, 0, 0, 0, 0, 0.5, 0.5]
        s2 = [0, 1, 0, 0, 0, 0, 0, 0]
        s3 = [0, 0, 0, 0, 0, 0, 0.5, 0.5]
        s4 = [0, 0, 0, 1, 0, 0, 0, 0]
        Value_cue_LR_0 = efe_al_ai_ex_ev(A, a, s1, 1, p_al, p_ai, p_ex) + efe_al_ai_ex_ev(A, a, s2, 1, p_al, p_ai, p_ex)
        Value_cue_LR_1 = efe_al_ai_ex_ev(A, a, s3, 1, p_al, p_ai, p_ex) + efe_al_ai_ex_ev(A, a, s4, 1, p_al, p_ai, p_ex)
        if Value_cue_LR_0[0]>Value_cue_LR_1[0]:
            Value_cue_LR = Value_cue_LR_1
        else:
            Value_cue_LR = Value_cue_LR_0
        Value_stay_cue.append((Value_cue_HR+Value_cue_LR)/2)
        # G_stay_cue = np.array([-min(G_stay_safe,G_stay_risk),-(G_cue_HR+G_cue_LR)/2])

        # exp_G = [np.exp(value_stay_cue[0]),np.exp(value_stay_cue[1])]
        # total = exp_G[0]+exp_G[1]
        # P = [exp_G[0]/total,exp_G[1]/total]
        return Value_stay_cue[action_stay_cue][0]*norm,Value_stay_cue[action_stay_cue][1]*norm,Value_stay_cue[action_stay_cue][2]*norm,Value_stay_cue[action_stay_cue][3]*norm,Value_stay_cue[action_stay_cue][4]*norm

def value_safe_risk(A,a,action_safe_risk,result_stay_cue,p_al,p_ai,p_ex,norm):
    Value_safe_risk = []
    if result_stay_cue !=0:
        action_stay_cue = 1
    else :
        action_stay_cue = 0
    if result_stay_cue == 0:
        s=np.array([0.5,0.5,0,0,0,0,0,0])
        Value_safe_risk.append(efe_al_ai_ex_ev(A,a,s,action_stay_cue,p_al,p_ai,p_ex))
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        Value_safe_risk.append(efe_al_ai_ex_ev(A,a,s,action_stay_cue,p_al,p_ai,p_ex))
    elif result_stay_cue == 1:
        s=np.array([1,0,0,0,0,0,0,0])
        Value_safe_risk.append(efe_al_ai_ex_ev(A,a,s,action_stay_cue,p_al,p_ai,p_ex))
        s=np.array([0,0,1,0,0,0,0,0])
        Value_safe_risk.append(efe_al_ai_ex_ev(A,a,s,action_stay_cue,p_al,p_ai,p_ex))
    else :
        s=np.array([0,1,0,0,0,0,0,0])
        Value_safe_risk.append(efe_al_ai_ex_ev(A,a,s,action_stay_cue,p_al,p_ai,p_ex))
        s=np.array([0,0,0,1,0,0,0,0])
        Value_safe_risk.append(efe_al_ai_ex_ev(A,a,s,action_stay_cue,p_al,p_ai,p_ex))

    return Value_safe_risk[action_safe_risk][0]*norm,Value_safe_risk[action_safe_risk][1]*norm,Value_safe_risk[action_safe_risk][2]*norm,Value_safe_risk[action_safe_risk][3]*norm,Value_safe_risk[action_safe_risk][4]*norm

def uncertainty_active_inference_stay_cue(A,a,action_stay_cue,result_stay_cue,if_can_ask,p_al,p_ai,p_ex,norm):

    if if_can_ask == 1:
        s1 = np.array([0,0,0,0,0,0,0.5,0.5])
        uncertainty_active_inference = ai(A,a,s1,1,p_al,p_ai,p_ex)
        return uncertainty_active_inference.squeeze()
    else:
        return ''


def result_epistemic_value_stay_cue(A,a,pi,con,can_not,p_al,p_ai,p_ex,nn):
    G_pi = []
    if can_not == 1:
        if pi == 0:
            s1 = np.array([0,0,0,0,0.5,0.5,0,0])
            s2 = np.array([0.5,0.5,0,0,0,0,0,0])
            G_pi.append((efe_al_ai_ex_ev(A,a,s1,0,p_al,p_ai,p_ex)+efe_al_ai_ex_ev(A,a,s2,0,p_al,p_ai,p_ex)))
            s1 = np.array([0,0,0,0,0.5,0.5,0,0])
            s2 = np.array([0,0,0.5,0.5,0,0,0,0])
            G_pi.append((efe_al_ai_ex_ev(A,a,s1,0,p_al,p_ai,p_ex)+efe_al_ai_ex_ev(A,a,s2,0,p_al,p_ai,p_ex)))
            if G_pi[0][0]<G_pi[1][0] :
                return float(G_pi[0][4])*nn
            else :
                return float(G_pi[1][4])*nn
        else :
            
            if con == 1:
                s1 = np.array([0,0,0,0,0,0,0.5,0.5])
                s2 = np.array([1,0,0,0,0,0,0,0])
                G_pi.append((efe_al_ai_ex_ev(A,a,s1,1,p_al,p_ai,p_ex)+efe_al_ai_ex_ev(A,a,s2,1,p_al,p_ai,p_ex)))
                s1 = np.array([0,0,0,0,0,0,0.5,0.5])
                s2 = np.array([0,0,1,0,0,0,0,0])
                G_pi.append((efe_al_ai_ex_ev(A,a,s1,1,p_al,p_ai,p_ex)+efe_al_ai_ex_ev(A,a,s2,1,p_al,p_ai,p_ex)))
                if G_pi[0][0]<G_pi[1][0] :
                    return float(G_pi[0][4])*nn
                else :
                    return float(G_pi[1][4])*nn
            elif con == 2:
                s1 = np.array([0,0,0,0,0,0,0.5,0.5])
                s2 = np.array([0,1,0,0,0,0,0,0])
                G_pi.append((efe_al_ai_ex_ev(A,a,s1,1,p_al,p_ai,p_ex)+efe_al_ai_ex_ev(A,a,s2,1,p_al,p_ai,p_ex)))
                s1 = np.array([0,0,0,0,0,0,0.5,0.5])
                s2 = np.array([0,0,0,1,0,0,0,0])
                G_pi.append((efe_al_ai_ex_ev(A,a,s1,1,p_al,p_ai,p_ex)+efe_al_ai_ex_ev(A,a,s2,1,p_al,p_ai,p_ex)))
                if G_pi[0][0]<G_pi[1][0] :
                    return float(G_pi[0][4])*nn
                else :
                    return float(G_pi[1][4])*nn
            else:
                print('ERROR-result_active_inference_stay_cue')
                raise ValueError('pi!=0,con=0')      
    else:
        return ''
    
def result_active_learning_safe_risk(A,a,action_safe_risk,result_stay_cue,result_safe_risk,p_al,p_ai,p_ex,norm):
    if result_safe_risk == 6:
        o = np.array([1,0,0,0,0,0,0,0])
    elif result_safe_risk == 12:
        o = np.array([0,1,0,0,0,0,0,0])
    elif result_safe_risk == 9:
        o = np.array([0,0,1,0,0,0,0,0])
    elif result_safe_risk == 3:
        o = np.array([0,0,0,1,0,0,0,0])
    elif result_safe_risk == 0:
        o = np.array([0,0,0,0,1,0,0,0])
    else :
        print('(result_safe_risk):',result_safe_risk)
        print('ERROR-result_active_learning_safe_risk')
    if action_safe_risk == 0 and result_stay_cue == 0:
        s=np.array([0.5,0.5,0,0,0,0,0,0])
    elif action_safe_risk ==0 and result_stay_cue ==1:
        s=np.array([1,0,0,0,0,0,0,0])
    elif action_safe_risk ==0 and result_stay_cue ==2:
        s=np.array([0,1,0,0,0,0,0,0])
    elif action_safe_risk ==1 and result_stay_cue ==0:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
    elif action_safe_risk ==1 and result_stay_cue ==1:
        s=np.array([0,0,1,0,0,0,0,0])
    elif action_safe_risk ==1 and result_stay_cue ==2:
        s=np.array([0,0,0,1,0,0,0,0])
    else:
        print('(result_stay_cue):',result_stay_cue)
        print('(action_safe_risk):',action_safe_risk)
        print('ERROR-result_active_learning_safe_risk')
    a_nonzero = np.array([[1,1,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,0,0,1,1,0,0],
                      [0,0,0,0,0,0,1,0],
                      [0,0,0,0,0,0,0,1]])
    w = 1./(cum(a))-1./(a+np.e**(-16))
    w = np.multiply(w,a_nonzero)
    AL = p_al*np.dot(o,np.dot(w,s.reshape(-1,1)))
    discount = 0.1
    if result_stay_cue ==0:
        return (discount*AL).squeeze()*norm
    else:
        return AL.squeeze()*norm
    
def uncertainty_active_learning_safe_risk(A,a,action_safe_risk,result_stay_cue,result_safe_risk,p_al,p_ai,p_ex,norm):
    if result_stay_cue !=0:
        action_stay_cue = 1
    else :
        action_stay_cue = 0
    if result_stay_cue == 0:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        uncertainty_safe_risk = al(A,a,s,action_stay_cue,p_al,p_ai,p_ex)
    elif result_stay_cue == 1:
        s=np.array([0,0,1,0,0,0,0,0])
        uncertainty_safe_risk = al(A,a,s,action_stay_cue,p_al,p_ai,p_ex)
    else :
        s=np.array([0,0,0,1,0,0,0,0])
        uncertainty_safe_risk = al(A,a,s,action_stay_cue,p_al,p_ai,p_ex)

    return uncertainty_safe_risk.squeeze()

    
def PE_active_learning_safe_risk(A,a,pi,con,reward,p_al,p_ai,p_ex,nn):
    expected = value_safe_risk(A,a,pi,con,p_al,p_ai,p_ex,nn)[1]
    result = result_active_learning_safe_risk(A,a,pi,con,reward,p_al,p_ai,p_ex,nn)
    return (result - expected).squeeze()
def PE_active_inference_stay_cue(A,a,pi,con,can_not,p_al,p_ai,p_ex,nn):
    if can_not ==0:
        return ''
    else:
        expected = value_stay_cue(A,a,pi,can_not,p_al,p_ai,p_ex,nn)[2]
        result = uncertainty_active_inference_stay_cue(A,a,pi,con,can_not,p_al,p_ai,p_ex,nn)
    return (result - expected).squeeze()

# def PE_epistemic_value_stay_cue(A,a,pi,con,can_not,p_al,p_ai,p_ex,nn):
#     if can_not ==0:
#         return ''
#     else:
#         expected = value_stay_cue(A,a,pi,can_not,p_al,p_ai,p_ex,nn)[4]
#         result = result_epistemic_value_stay_cue(A,a,pi,con,can_not,p_al,p_ai,p_ex,nn)
#     return result - expected

def PE_reward(A,a,pi,con,reward,p_al,p_ai,p_ex,nn):
    if pi == 0 and con == 0:
        s=np.array([0.5,0.5,0,0,0,0,0,0])
    elif pi ==0 and con ==1:
        s=np.array([1,0,0,0,0,0,0,0])
    elif pi ==0 and con ==2:
        s=np.array([0,1,0,0,0,0,0,0])
    elif pi ==1 and con ==0:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
    elif pi ==1 and con ==1:
        s=np.array([0,0,1,0,0,0,0,0])
    elif pi ==1 and con ==2:
        s=np.array([0,0,0,1,0,0,0,0])
    else:
        print('ERROR-PE_reward')
    o = np.dot(A,s.reshape(-1,1)).reshape(-1)
    preference = np.array([1,2,1.5,0.5,0,0,-0.167,-0.167])
    EX = np.dot(o,preference.reshape(-1,1))
    return (reward/6 -EX).squeeze()

def a_update(a,result_stay_cue,result_safe_risk,action_safe_risk,rate):
    if result_stay_cue==0 and result_safe_risk==6 and action_safe_risk==0:
        s=np.array([0.5,0.5,0,0,0,0,0,0])
#safeHRC,safeLRC,riskyHRC,riskLRC,stayHRC,stayLRC,cueHRC,cueLRC
        o=np.array([1,0,0,0,0,0,0,0])
#safe,riskyHR,riskyLR,stay,cueHR,cueLR
    elif result_stay_cue==0 and result_safe_risk==0:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([0,0,0,0,1,0,0,0])
    elif result_stay_cue==0 and result_safe_risk==3:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([0,0,0,1,0,0,0,0])
    elif result_stay_cue==0 and result_safe_risk==6 and action_safe_risk==1:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==0 and result_safe_risk==9:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([0,0,1,0,0,0,0,0])
    elif result_stay_cue==0 and result_safe_risk==12:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([0,1,0,0,0,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==6 and action_safe_risk==0:
        s=np.array([1,0,0,0,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==0:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([0,0,0,0,1,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==3:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([0,0,0,1,0,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==6 and action_safe_risk==1:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==9:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([0,0,1,0,0,0,0,0])      
    elif result_stay_cue==1 and result_safe_risk==12:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([0,1,0,0,0,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==6 and action_safe_risk==0:
        s=np.array([0,1,0,0,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==0:
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([0,0,0,0,1,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==3:
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([0,0,0,1,0,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==6 and action_safe_risk==1:
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==9:
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([0,0,1,0,0,0,0,0])    
    else :
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([0,1,0,0,0,0,0,0])
    if result_stay_cue!=0:
        a=a+rate*np.outer(o,s)#rate:learning rate
    else :
        a=a+rate*0.1*np.outer(o,s)
    return a

def PE_epistemic_value_stay_cue(A,a,pi,con,can_not,p_al,p_ai,p_ex,nn):
    if can_not ==0:
        return ''
    else:
        expected = value_stay_cue(A,a,pi,can_not,p_al,p_ai,p_ex,nn)[4]
        result = result_epistemic_value_stay_cue(A,a,pi,con,can_not,p_al,p_ai,p_ex,nn)
    return result - expected

def read_behavioral_data(n):
    result_stay_cue = []
    result_safe_risk = []
    action_stay_cue = []
    action_safe_risk = []
    if_can_ask = []
    fname = './behavioral_data/uncertainty_' + str(n+1) + '_2022.csv'
    with open(fname,'r') as f :
        for line in f.readlines():
            if line.split(',')[0]=='0' or line.split(',')[0]=='1':
                if int(line.split(',')[3])==0:
                    action_stay_cue.append(0)
                elif int(line.split(',')[3])==1 or int(line.split(',')[3])==2:
                    action_stay_cue.append(1)
                else :
                    print('ERROR')
                result_stay_cue.append(int(line.split(',')[3]))
                result_safe_risk.append(int(float(line.split(',')[6])))
                action_safe_risk.append(int(line.split(',')[4]))
                if line.split(',')[1]==' ':
                    if_can_ask.append(0)
                else :
                    if_can_ask.append(1)

    return if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk

def regressor_choosing_stay_cue(sub,rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(sub)
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                  [0,0,prior,prior,0,0,0,0],
                  [0,0,prior,prior,0,0,0,0],
                  [0,0,prior,prior,0,0,0,0],
                  [0,0,prior,prior,0,0,0,0],
                  [0,0,0,0,100,100,0,0],
                  [0,0,0,0,0,0,100,0],
                  [0,0,0,0,0,0,0,100]])
    A = dir(a)
    EF_1,AL_1,AI_1,EX_1,EV_1 = [],[],[],[],[]
    for i in range(trial_num):
        ef_1,al_1,ai_1,ex_1,ev_1 = value_stay_cue(A,a,action_stay_cue[i],if_can_ask[i],p_al,p_ai,p_ex,1/(p_ex+np.e**(-16)))
        EF_1.append(ef_1)
        AL_1.append(al_1)
        AI_1.append(ai_1)
        EX_1.append(ex_1)
        EV_1.append(ev_1)
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return EF_1,AL_1,AI_1,EX_1,EV_1

def regressor_choosing_safe_risk(sub,rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(sub)
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                  [0,0,prior,prior,0,0,0,0],
                  [0,0,prior,prior,0,0,0,0],
                  [0,0,prior,prior,0,0,0,0],
                  [0,0,prior,prior,0,0,0,0],
                  [0,0,0,0,100,100,0,0],
                  [0,0,0,0,0,0,100,0],
                  [0,0,0,0,0,0,0,100]])
    A = dir(a)
    EF_2,AL_2,AI_2,EX_2,EV_2 = [],[],[],[],[]
    for i in range(trial_num):
        ef_2,al_2,ai_2,ex_2,ev_2 = value_safe_risk(A,a,action_stay_cue[i],if_can_ask[i],p_al,p_ai,p_ex,1/(p_ex+np.e**(-16)))
        EF_2.append(ef_2)
        AL_2.append(al_2)
        AI_2.append(ai_2)
        EX_2.append(ex_2)
        EV_2.append(ev_2)
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return EF_2,AL_2,AI_2,EX_2,EV_2

def regressor_result_uncertainty(sub,rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(sub)
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                  [0,0,prior,prior,0,0,0,0],
                  [0,0,prior,prior,0,0,0,0],
                  [0,0,prior,prior,0,0,0,0],
                  [0,0,prior,prior,0,0,0,0],
                  [0,0,0,0,100,100,0,0],
                  [0,0,0,0,0,0,100,0],
                  [0,0,0,0,0,0,0,100]])
    A = dir(a)
    uncertainty_AI,uncertainty_AL,result_AL,result_EX = [],[],[],[]
    for i in range(trial_num):
        uncertainty_AI.append(uncertainty_active_inference_stay_cue(A,a,action_stay_cue[i],result_stay_cue[i],if_can_ask[i],p_al,p_ai,p_ex,1/(p_ex+np.e**(-16))))
        uncertainty_AL.append(uncertainty_active_learning_safe_risk(A,a,action_safe_risk[i],result_stay_cue[i],result_safe_risk[i],p_al,p_ai,p_ex,1/(p_ex+np.e**(-16))))
        result_AL.append(result_active_learning_safe_risk(A,a,action_safe_risk[i],result_stay_cue[i],result_safe_risk[i],p_al,p_ai,p_ex,1/(p_ex+np.e**(-16))))
        result_EX.append(result_safe_risk[i]/6)
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return uncertainty_AI,uncertainty_AL,result_AL,result_EX

def regressor_prediction_error(sub,rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(sub)
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                  [0,0,prior,prior,0,0,0,0],
                  [0,0,prior,prior,0,0,0,0],
                  [0,0,prior,prior,0,0,0,0],
                  [0,0,prior,prior,0,0,0,0],
                  [0,0,0,0,100,100,0,0],
                  [0,0,0,0,0,0,100,0],
                  [0,0,0,0,0,0,0,100]])
    A = dir(a)
    pe_AL,pe_AI,pe_EX,pe_EV = [],[],[],[]
    for i in range(trial_num):
        pe_AL.append(PE_active_learning_safe_risk(A,a,action_safe_risk[i],result_stay_cue[i],result_safe_risk[i],p_al,p_ai,p_ex,1/(p_ex+np.e**(-16))))
        pe_AI.append(PE_active_inference_stay_cue(A,a,action_stay_cue[i],result_stay_cue[i],if_can_ask[i],p_al,p_ai,p_ex,1/(p_ex+np.e**(-16))))
        pe_EX.append(PE_reward(A,a,action_safe_risk[i],result_stay_cue[i],result_safe_risk[i],p_al,p_ai,p_ex,1/(p_ex+np.e**(-16))))
        pe_EV.append(PE_epistemic_value_stay_cue(A,a,action_stay_cue[i],result_stay_cue[i],if_can_ask[i],p_al,p_ai,p_ex,1/(p_ex+np.e**(-16))))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return pe_AL,pe_AI,pe_EX,pe_EV



In [3]:
def dir(a):
    a_0 = np.sum(a,axis=0)
    A = np.zeros([8,8])
    for i in range(np.shape(A)[0]):
        for j in range(np.shape(A)[1]):
            A[i,j] = a[i,j]/a_0[j]
    return A
def cum(a):
    a_0 = np.sum(a,axis=0)
    a_cum = np.array([np.ones([1,8])*a_0[0],
                      np.ones([1,8])*a_0[1],
                      np.ones([1,8])*a_0[2],
                      np.ones([1,8])*a_0[3],
                      np.ones([1,8])*a_0[4],
                      np.ones([1,8])*a_0[5],
                      np.ones([1,8])*a_0[6],
                      np.ones([1,8])*a_0[7]]).squeeze().T
    return a_cum
def H_entropy(A):
    H = np.matmul(A.T,np.log(A+np.e**(-16)))
    H = np.diag(H)
    return H
def G_ExperctedFreeEnergy(A,a,s,action_stay_cue,p_al,p_ai,p_ex):
    a_nonzero = np.array([[1,1,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,0,0,1,1,0,0],
                      [0,0,0,0,0,0,1,0],
                      [0,0,0,0,0,0,0,1]])
    s = np.array(s)
    discount = 0.1
    o = np.dot(A,s.reshape(-1,1)).reshape(-1)
    w = 1./(cum(a))-1./(a+np.e**(-16))
    w = np.multiply(w,a_nonzero)
    H = -np.dot(H_entropy(A),s.reshape(-1,1))
    AL = np.dot(o,np.dot(w,s.reshape(-1,1)))
    AI = H + np.dot(o,np.log(o+np.e**(-16)).reshape(-1,1))
    preference = np.array([1,2,1.5,0.5,0,0,-0.167,-0.167])
    EX = np.dot(o,preference.reshape(-1,1))
    if action_stay_cue == 0:
        return float(discount*(p_al*AL + p_ai*AI) - p_ex*EX)
    elif action_stay_cue == 1:
        return float(p_al*AL + p_ai*AI - p_ex*EX)
    else :
        print('ERROR')
        
def P_stay_cue(A,a,action_stay_cue,if_can_ask,p_al,p_ai,p_ex):#stay,cue,0,stay-safe,1,stay-risk,2,cue-safe,3,cue-risk
    if if_can_ask == 0:
        return 1
    else:
        s1 = np.array([0,0,0,0,0.5,0.5,0,0])
        s2 = np.array([0.5,0.5,0,0,0,0,0,0])
        G_stay_safe = G_ExperctedFreeEnergy(A,a,s1,0,p_al,p_ai,p_ex)+G_ExperctedFreeEnergy(A,a,s2,0,p_al,p_ai,p_ex)
        s1 = np.array([0,0,0,0,0.5,0.5,0,0])
        s2 = np.array([0,0,0.5,0.5,0,0,0,0])
        G_stay_risk = G_ExperctedFreeEnergy(A,a,s1,0,p_al,p_ai,p_ex)+G_ExperctedFreeEnergy(A,a,s2,0,p_al,p_ai,p_ex)
        s1 = [0, 0, 0, 0, 0, 0, 0.5, 0.5]
        s2 = [1, 0, 0, 0, 0, 0, 0, 0]
        s3 = [0, 0, 0, 0, 0, 0, 0.5, 0.5]
        s4 = [0, 0, 1, 0, 0, 0, 0, 0]
        G_cue_HR_0 = G_ExperctedFreeEnergy(A, a, s1, 1, p_al, p_ai, p_ex) + G_ExperctedFreeEnergy(A, a, s2, 1, p_al, p_ai, p_ex)
        G_cue_HR_1 = G_ExperctedFreeEnergy(A, a, s3, 1, p_al, p_ai, p_ex) + G_ExperctedFreeEnergy(A, a, s4, 1, p_al, p_ai, p_ex)
        if G_cue_HR_0>G_cue_HR_1:
            G_cue_HR = G_cue_HR_1
        else:
            G_cue_HR = G_cue_HR_0
        s1 = [0, 0, 0, 0, 0, 0, 0.5, 0.5]
        s2 = [0, 1, 0, 0, 0, 0, 0, 0]
        s3 = [0, 0, 0, 0, 0, 0, 0.5, 0.5]
        s4 = [0, 0, 0, 1, 0, 0, 0, 0]
        G_cue_LR_0 = G_ExperctedFreeEnergy(A, a, s1, 1, p_al, p_ai, p_ex) + G_ExperctedFreeEnergy(A, a, s2, 1, p_al, p_ai, p_ex)
        G_cue_LR_1 = G_ExperctedFreeEnergy(A, a, s3, 1, p_al, p_ai, p_ex) + G_ExperctedFreeEnergy(A, a, s4, 1, p_al, p_ai, p_ex)
        if G_cue_LR_0>G_cue_LR_1:
            G_cue_LR = G_cue_LR_1
        else:
            G_cue_LR = G_cue_LR_0
        G_stay_cue = np.array([-min(G_stay_safe,G_stay_risk),-(G_cue_HR+G_cue_LR)/2])
        if G_stay_cue[0]>20:
            G_stay_cue[0]=20
        if G_stay_cue[1]>20:
            G_stay_cue[1]=20
        exp_G = [np.exp(G_stay_cue[0]),np.exp(G_stay_cue[1])]
        total = exp_G[0]+exp_G[1]
        P = [exp_G[0]/total,exp_G[1]/total]
        return P[action_stay_cue].squeeze()

def P_safe_risk(A,a,action_safe_risk,result_stay_cue,p_al,p_ai,p_ex):#con=0,no,con=1,HRC,con=2,LRC,pi=0,safe,pi=1,risky
    if result_stay_cue !=0:
        action_stay_cue = 1
    else :
        action_stay_cue = 0
    if result_stay_cue == 0:
        s=np.array([0.5,0.5,0,0,0,0,0,0])
        G_safe = -G_ExperctedFreeEnergy(A,a,s,action_stay_cue,p_al,p_ai,p_ex)
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        G_risk = -G_ExperctedFreeEnergy(A,a,s,action_stay_cue,p_al,p_ai,p_ex)
    elif result_stay_cue == 1:
        s=np.array([1,0,0,0,0,0,0,0])
        G_safe = -G_ExperctedFreeEnergy(A,a,s,action_stay_cue,p_al,p_ai,p_ex)
        s=np.array([0,0,1,0,0,0,0,0])
        G_risk = -G_ExperctedFreeEnergy(A,a,s,action_stay_cue,p_al,p_ai,p_ex)
    else :
        s=np.array([0,1,0,0,0,0,0,0])
        G_safe = -G_ExperctedFreeEnergy(A,a,s,action_stay_cue,p_al,p_ai,p_ex)
        s=np.array([0,0,0,1,0,0,0,0])
        G_risk = -G_ExperctedFreeEnergy(A,a,s,action_stay_cue,p_al,p_ai,p_ex)
    if G_safe > 20:
        G_safe = 20
    if G_risk > 20:
        G_risk = 20
    exp_G = [np.exp(G_safe),np.exp(G_risk)]
    total = exp_G[0]+exp_G[1]
    P_safe_risk = [exp_G[0]/total,exp_G[1]/total]
    return P_safe_risk[action_safe_risk].squeeze()

def a_update(a,result_stay_cue,result_safe_risk,action_safe_risk,rate):
    if result_stay_cue==0 and result_safe_risk==6 and action_safe_risk==0:
        s=np.array([0.5,0.5,0,0,0,0,0,0])
#safeHRC,safeLRC,riskyHRC,riskLRC,stayHRC,stayLRC,cueHRC,cueLRC
        o=np.array([1,0,0,0,0,0,0,0])
#safe,riskyHR,riskyLR,stay,cueHR,cueLR
    elif result_stay_cue==0 and result_safe_risk==0:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([0,0,0,0,1,0,0,0])
    elif result_stay_cue==0 and result_safe_risk==3:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([0,0,0,1,0,0,0,0])
    elif result_stay_cue==0 and result_safe_risk==6 and action_safe_risk==1:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==0 and result_safe_risk==9:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([0,0,1,0,0,0,0,0])
    elif result_stay_cue==0 and result_safe_risk==12:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([0,1,0,0,0,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==6 and action_safe_risk==0:
        s=np.array([1,0,0,0,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==0:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([0,0,0,0,1,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==3:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([0,0,0,1,0,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==6 and action_safe_risk==1:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==9:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([0,0,1,0,0,0,0,0])      
    elif result_stay_cue==1 and result_safe_risk==12:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([0,1,0,0,0,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==6 and action_safe_risk==0:
        s=np.array([0,1,0,0,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==0:
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([0,0,0,0,1,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==3:
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([0,0,0,1,0,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==6 and action_safe_risk==1:
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==9:
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([0,0,1,0,0,0,0,0])    
    else :
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([0,1,0,0,0,0,0,0])
    if result_stay_cue!=0:
        a=a+rate*np.outer(o,s)#rate:learning rate
    else :
        a=a+rate*0.1*np.outer(o,s)
    return a

def P_active_inference(x):
    trial_num = 120
    subject_num = 25
    if_can_ask_sub = []
    action_stay_cue_sub = []
    result_stay_cue_sub = []
    action_safe_risk_sub = []
    result_safe_risk_sub = []
    for i in range(subject_num):
        if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(i)
        if_can_ask_sub.append(if_can_ask)
        action_stay_cue_sub.append(action_stay_cue)
        result_stay_cue_sub.append(result_stay_cue)
        action_safe_risk_sub.append(action_safe_risk)
        result_safe_risk_sub.append(result_safe_risk)
    rate = []
    prior = []
    p_al = []
    p_ai = []
    p_ex = []
    for i in range(subject_num):
        rate.append(x[i*5])
        prior.append(x[i*5+1])
        p_al.append(x[i*5+2])
        p_ai.append(x[i*5+3])
        p_ex.append(x[i*5+4])
    neg_log_p_policy = 0
    for i in range(subject_num):
        a = np.array([[100.0,100,prior[i],prior[i],0,0,0,0],
                  [0,0,prior[i],prior[i],0,0,0,0],
                  [0,0,prior[i],prior[i],0,0,0,0],
                  [0,0,prior[i],prior[i],0,0,0,0],
                  [0,0,prior[i],prior[i],0,0,0,0],
                  [0,0,0,0,100,100,0,0],
                  [0,0,0,0,0,0,100,0],
                  [0,0,0,0,0,0,0,100]])
        A = dir(a)
        for j in range(trial_num):
            neg_log_p_policy -= np.log(P_stay_cue(A,a,action_stay_cue_sub[i][j],if_can_ask_sub[i][j],p_al[i],p_ai[i],p_ex[i]))
            neg_log_p_policy -= np.log(P_safe_risk(A,a,action_safe_risk_sub[i][j],result_stay_cue_sub[i][j],p_al[i],p_ai[i],p_ex[i]))
            a = a_update(a,result_stay_cue_sub[i][j],result_safe_risk_sub[i][j],action_safe_risk_sub[i][j],rate[i])
            A = dir(a)
    return neg_log_p_policy

In [4]:
def P_active_inference_subject_1(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 0

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_2(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 1

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_3(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 2

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_4(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 3

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_5(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 4

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_6(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 5

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_7(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 6

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_8(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 7

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_9(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 8

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_10(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 9

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_11(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 10

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_12(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 11

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_13(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 12

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_14(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 13

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_15(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 14

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_16(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 15

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_17(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 16

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_18(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 17

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_19(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 18

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_20(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 19

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_21(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 20

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_22(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 21
    
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_23(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 22
    
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_24(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 23
    
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_25(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 24
    
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy

In [5]:
P_AL,P_AI,P_EX,PRIOR,RATE,LL = [],[],[],[],[],[]

pbounds = {'rate':(0.001,10),'prior':(0.001,10),'p_al':(0.001,10),'p_ai':(0.001,10),'p_ex':(0.001,10)}
optimizer_1 = BayesianOptimization(
    f=P_active_inference_subject_1,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_1, logs=["./logs_1.log.json"])
P_AL.append(optimizer_1.max['params']['p_al'])
P_AI.append(optimizer_1.max['params']['p_ai'])
P_EX.append(optimizer_1.max['params']['p_ex'])
RATE.append(optimizer_1.max['params']['rate'])
PRIOR.append(optimizer_1.max['params']['prior'])
LL.append(optimizer_1.max['target'])

# optimizer_2 = BayesianOptimization(
#     f=P_active_inference_subject_2,
#     pbounds=pbounds,
#     random_state=1,allow_duplicate_points=True)
# load_logs(optimizer_2, logs=["./logs_2.log.json"])
# "p_ai": 0.6976539349355139, "p_al": 1.0808592298701158, "p_ex": 7.164060397933428, "prior": 0.2649931532879722, "rate": 5.396963570672858
P_AL.append(1.0808592298701158)
P_AI.append(0.6976539349355139)
P_EX.append(7.164060397933428)
RATE.append(5.396963570672858)
PRIOR.append(0.2649931532879722)
LL.append(-96.47016096446688)

optimizer_3 = BayesianOptimization(
    f=P_active_inference_subject_3,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_3, logs=["./logs_3.log.json"])
P_AL.append(optimizer_3.max['params']['p_al'])
P_AI.append(optimizer_3.max['params']['p_ai'])
P_EX.append(optimizer_3.max['params']['p_ex'])
RATE.append(optimizer_3.max['params']['rate'])
PRIOR.append(optimizer_3.max['params']['prior'])
LL.append(optimizer_3.max['target'])

optimizer_4 = BayesianOptimization(
    f=P_active_inference_subject_4,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_4, logs=["./logs_4.log.json"])
P_AL.append(optimizer_4.max['params']['p_al'])
P_AI.append(optimizer_4.max['params']['p_ai'])
P_EX.append(optimizer_4.max['params']['p_ex'])
RATE.append(optimizer_4.max['params']['rate'])
PRIOR.append(optimizer_4.max['params']['prior'])
LL.append(optimizer_4.max['target'])

optimizer_5 = BayesianOptimization(
    f=P_active_inference_subject_5,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_5, logs=["./logs_5.log.json"])
P_AL.append(optimizer_5.max['params']['p_al'])
P_AI.append(optimizer_5.max['params']['p_ai'])
P_EX.append(optimizer_5.max['params']['p_ex'])
RATE.append(optimizer_5.max['params']['rate'])
PRIOR.append(optimizer_5.max['params']['prior'])
LL.append(optimizer_5.max['target'])

optimizer_6 = BayesianOptimization(
    f=P_active_inference_subject_6,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_6, logs=["./logs_6.log.json"])
P_AL.append(optimizer_6.max['params']['p_al'])
P_AI.append(optimizer_6.max['params']['p_ai'])
P_EX.append(optimizer_6.max['params']['p_ex'])
RATE.append(optimizer_6.max['params']['rate'])
PRIOR.append(optimizer_6.max['params']['prior'])
LL.append(optimizer_6.max['target'])

optimizer_7 = BayesianOptimization(
    f=P_active_inference_subject_7,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_7, logs=["./logs_7.log.json"])
P_AL.append(optimizer_7.max['params']['p_al'])
P_AI.append(optimizer_7.max['params']['p_ai'])
P_EX.append(optimizer_7.max['params']['p_ex'])
RATE.append(optimizer_7.max['params']['rate'])
PRIOR.append(optimizer_7.max['params']['prior'])
LL.append(optimizer_7.max['target'])

optimizer_8 = BayesianOptimization(
    f=P_active_inference_subject_8,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_8, logs=["./logs_8.log.json"])
P_AL.append(optimizer_8.max['params']['p_al'])
P_AI.append(optimizer_8.max['params']['p_ai'])
P_EX.append(optimizer_8.max['params']['p_ex'])
RATE.append(optimizer_8.max['params']['rate'])
PRIOR.append(optimizer_8.max['params']['prior'])
LL.append(optimizer_8.max['target'])

optimizer_9 = BayesianOptimization(
    f=P_active_inference_subject_9,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_9, logs=["./logs_9.log.json"])
P_AL.append(optimizer_9.max['params']['p_al'])
P_AI.append(optimizer_9.max['params']['p_ai'])
P_EX.append(optimizer_9.max['params']['p_ex'])
RATE.append(optimizer_9.max['params']['rate'])
PRIOR.append(optimizer_9.max['params']['prior'])
LL.append(optimizer_9.max['target'])

optimizer_10 = BayesianOptimization(
    f=P_active_inference_subject_10,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_10, logs=["./logs_10.log.json"])
P_AL.append(optimizer_10.max['params']['p_al'])
P_AI.append(optimizer_10.max['params']['p_ai'])
P_EX.append(optimizer_10.max['params']['p_ex'])
RATE.append(optimizer_10.max['params']['rate'])
PRIOR.append(optimizer_10.max['params']['prior'])
LL.append(optimizer_10.max['target'])

optimizer_11 = BayesianOptimization(
    f=P_active_inference_subject_11,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_11, logs=["./logs_11.log.json"])
P_AL.append(optimizer_11.max['params']['p_al'])
P_AI.append(optimizer_11.max['params']['p_ai'])
P_EX.append(optimizer_11.max['params']['p_ex'])
RATE.append(optimizer_11.max['params']['rate'])
PRIOR.append(optimizer_11.max['params']['prior'])
LL.append(optimizer_11.max['target'])

optimizer_12 = BayesianOptimization(
    f=P_active_inference_subject_12,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_12, logs=["./logs_12.log.json"])
P_AL.append(optimizer_12.max['params']['p_al'])
P_AI.append(optimizer_12.max['params']['p_ai'])
P_EX.append(optimizer_12.max['params']['p_ex'])
RATE.append(optimizer_12.max['params']['rate'])
PRIOR.append(optimizer_12.max['params']['prior'])
LL.append(optimizer_12.max['target'])

optimizer_13 = BayesianOptimization(
    f=P_active_inference_subject_13,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_13, logs=["./logs_13.log.json"])
P_AL.append(optimizer_13.max['params']['p_al'])
P_AI.append(optimizer_13.max['params']['p_ai'])
P_EX.append(optimizer_13.max['params']['p_ex'])
RATE.append(optimizer_13.max['params']['rate'])
PRIOR.append(optimizer_13.max['params']['prior'])
LL.append(optimizer_13.max['target'])

optimizer_14 = BayesianOptimization(
    f=P_active_inference_subject_14,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_14, logs=["./logs_14.log.json"])
P_AL.append(optimizer_14.max['params']['p_al'])
P_AI.append(optimizer_14.max['params']['p_ai'])
P_EX.append(optimizer_14.max['params']['p_ex'])
RATE.append(optimizer_14.max['params']['rate'])
PRIOR.append(optimizer_14.max['params']['prior'])
LL.append(optimizer_14.max['target'])

optimizer_15 = BayesianOptimization(
    f=P_active_inference_subject_15,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_15, logs=["./logs_15.log.json"])
P_AL.append(optimizer_15.max['params']['p_al'])
P_AI.append(optimizer_15.max['params']['p_ai'])
P_EX.append(optimizer_15.max['params']['p_ex'])
RATE.append(optimizer_15.max['params']['rate'])
PRIOR.append(optimizer_15.max['params']['prior'])
LL.append(optimizer_15.max['target'])

optimizer_16 = BayesianOptimization(
    f=P_active_inference_subject_16,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_16, logs=["./logs_16.log.json"])
P_AL.append(optimizer_16.max['params']['p_al'])
P_AI.append(optimizer_16.max['params']['p_ai'])
P_EX.append(optimizer_16.max['params']['p_ex'])
RATE.append(optimizer_16.max['params']['rate'])
PRIOR.append(optimizer_16.max['params']['prior'])
LL.append(optimizer_16.max['target'])

optimizer_17 = BayesianOptimization(
    f=P_active_inference_subject_17,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_17, logs=["./logs_17.log.json"])
P_AL.append(optimizer_17.max['params']['p_al'])
P_AI.append(optimizer_17.max['params']['p_ai'])
P_EX.append(optimizer_17.max['params']['p_ex'])
RATE.append(optimizer_17.max['params']['rate'])
PRIOR.append(optimizer_17.max['params']['prior'])
LL.append(optimizer_17.max['target'])

optimizer_18 = BayesianOptimization(
    f=P_active_inference_subject_18,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_18, logs=["./logs_18.log.json"])
P_AL.append(optimizer_18.max['params']['p_al'])
P_AI.append(optimizer_18.max['params']['p_ai'])
P_EX.append(optimizer_18.max['params']['p_ex'])
RATE.append(optimizer_18.max['params']['rate'])
PRIOR.append(optimizer_18.max['params']['prior'])
LL.append(optimizer_18.max['target'])

optimizer_19 = BayesianOptimization(
    f=P_active_inference_subject_19,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_19, logs=["./logs_19.log.json"])
P_AL.append(optimizer_19.max['params']['p_al'])
P_AI.append(optimizer_19.max['params']['p_ai'])
P_EX.append(optimizer_19.max['params']['p_ex'])
RATE.append(optimizer_19.max['params']['rate'])
PRIOR.append(optimizer_19.max['params']['prior'])
LL.append(optimizer_19.max['target'])

optimizer_20 = BayesianOptimization(
    f=P_active_inference_subject_20,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_20, logs=["./logs_20.log.json"])
P_AL.append(optimizer_20.max['params']['p_al'])
P_AI.append(optimizer_20.max['params']['p_ai'])
P_EX.append(optimizer_20.max['params']['p_ex'])
RATE.append(optimizer_20.max['params']['rate'])
PRIOR.append(optimizer_20.max['params']['prior'])
LL.append(optimizer_20.max['target'])

optimizer_21 = BayesianOptimization(
    f=P_active_inference_subject_21,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_21, logs=["./logs_21.log.json"])
P_AL.append(optimizer_21.max['params']['p_al'])
P_AI.append(optimizer_21.max['params']['p_ai'])
P_EX.append(optimizer_21.max['params']['p_ex'])
RATE.append(optimizer_21.max['params']['rate'])
PRIOR.append(optimizer_21.max['params']['prior'])
LL.append(optimizer_21.max['target'])

optimizer_22 = BayesianOptimization(
    f=P_active_inference_subject_22,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_22, logs=["./logs_22.log.json"])
P_AL.append(optimizer_22.max['params']['p_al'])
P_AI.append(optimizer_22.max['params']['p_ai'])
P_EX.append(optimizer_22.max['params']['p_ex'])
RATE.append(optimizer_22.max['params']['rate'])
PRIOR.append(optimizer_22.max['params']['prior'])
LL.append(optimizer_22.max['target'])

optimizer_23 = BayesianOptimization(
    f=P_active_inference_subject_23,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_23, logs=["./logs_23.log.json"])
P_AL.append(optimizer_23.max['params']['p_al'])
P_AI.append(optimizer_23.max['params']['p_ai'])
P_EX.append(optimizer_23.max['params']['p_ex'])
RATE.append(optimizer_23.max['params']['rate'])
PRIOR.append(optimizer_23.max['params']['prior'])
LL.append(optimizer_23.max['target'])

optimizer_24 = BayesianOptimization(
    f=P_active_inference_subject_24,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_24, logs=["./logs_24.log.json"])
P_AL.append(optimizer_24.max['params']['p_al'])
P_AI.append(optimizer_24.max['params']['p_ai'])
P_EX.append(optimizer_24.max['params']['p_ex'])
RATE.append(optimizer_24.max['params']['rate'])
PRIOR.append(optimizer_24.max['params']['prior'])
LL.append(optimizer_24.max['target'])

optimizer_25 = BayesianOptimization(
    f=P_active_inference_subject_25,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_25, logs=["./logs_25.log.json"])
P_AL.append(optimizer_25.max['params']['p_al'])
P_AI.append(optimizer_25.max['params']['p_ai'])
P_EX.append(optimizer_25.max['params']['p_ex'])
RATE.append(optimizer_25.max['params']['rate'])
PRIOR.append(optimizer_25.max['params']['prior'])
LL.append(optimizer_25.max['target'])


Data point [1.e-03 1.e-03 1.e+01 1.e+01 1.e-03] is not unique. 1 duplicates registered. Continuing ...
Data point [1.e-03 1.e-03 1.e+01 1.e+01 1.e-03] is not unique. 2 duplicates registered. Continuing ...
Data point [1.e-03 1.e-03 1.e+01 1.e+01 1.e-03] is not unique. 3 duplicates registered. Continuing ...
Data point [1.e-03 1.e-03 1.e+01 1.e+01 1.e-03] is not unique. 4 duplicates registered. Continuing ...
Data point [1.e-03 1.e-03 1.e+01 1.e+01 1.e-03] is not unique. 5 duplicates registered. Continuing ...
Data point [1.e-03 1.e-03 1.e+01 1.e+01 1.e-03] is not unique. 6 duplicates registered. Continuing ...
Data point [1.e-03 1.e-03 1.e+01 1.e+01 1.e-03] is not unique. 7 duplicates registered. Continuing ...
Data point [1.e-03 1.e-03 1.e+01 1.e+01 1.e-03] is not unique. 8 duplicates registered. Continuing ...
Data point [1.e-03 1.e-03 1.e+01 1.e+01 1.e-03] is not unique. 9 duplicates registered. Continuing ...
Data point [1.e-03 1.e-03 1.e+01 1.e+01 1.e-03] is not unique. 10 duplica

In [7]:
for i in range(25):
    subject = i
    rate = RATE[i]
    prior = PRIOR[i]
    p_al = P_AL[i]
    p_ai = P_AI[i]
    p_ex = P_EX[i]
    EF_stay_cue,AL_stay_cue,AI_stay_cue,EX_stay_cue,EV_stay_cue = regressor_choosing_stay_cue(i,rate,prior,p_al,p_ai,p_ex)
    EF_safe_risk,AL_safe_risk,AI_safe_risk,EX_safe_risk,EV_safe_risk = regressor_choosing_safe_risk(i,rate,prior,p_al,p_ai,p_ex)
    uncertainty_AI,uncertainty_AL,result_AL,result_EX = regressor_result_uncertainty(i,rate,prior,p_al,p_ai,p_ex)
    prediciton_error_AL_safe_risk,prediciton_error_AI_stay_cue,prediciton_error_EX,prediction_error_EV_stay_cue = regressor_prediction_error(i,rate,prior,p_al,p_ai,p_ex)  
    file_name = 'subject'+str(subject+1)+'_regress_data.csv'
    with open(file_name, 'w', encoding='utf-8', newline='') as f: 
        write = csv.writer(f)  # 创建writer对象
        write.writerow(['EFE1','AL1','AI1','EX1','EV1',
                        'EFE2','AL2','AI2','EX2','EV2',
                        'UAI','UAL','RAL','REX',
                        'PEAL2','PEAI1','PEEX2','PEEV1'])
        for i in range(120):
            write.writerow([EF_stay_cue[i],AL_stay_cue[i],AI_stay_cue[i],EX_stay_cue[i],EV_stay_cue[i],
                            EF_safe_risk[i],AL_safe_risk[i],AI_safe_risk[i],EX_safe_risk[i],EV_safe_risk[i],
                            uncertainty_AI[i],uncertainty_AL[i],result_AL[i],result_EX[i],
                            prediciton_error_AL_safe_risk[i],prediciton_error_AI_stay_cue[i],prediciton_error_EX[i],prediction_error_EV_stay_cue[i]])